# Virtual Manipulation of GRIB2 with VirtualiZarr and grib2io

This notebook demonstrates how to use `VirtualiZarr` to open a grib2io kerchunk manifest
as a virtual dataset — holding only metadata references, no data bytes — and how to
load actual data via `grib2io.icechunk.open_grib2`.


In [ ]:
import xarray as xr
import grib2io
import json
import pathlib
import os
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import KerchunkJSONParser
from obstore.store import LocalStore
from obspec_utils.registry import ObjectStoreRegistry
from grib2io.kerchunk import ReferenceGenerator
from grib2io.icechunk import open_grib2, open_dataset

## 1. Build a Kerchunk Manifest

`ReferenceGenerator` scans a GRIB2 file and produces a kerchunk v1 reference manifest —
a plain dict mapping variable/coordinate names to byte-range references. No data is read.


In [ ]:
_here = pathlib.Path(os.path.abspath(""))
_candidates = [
    _here / "tests" / "input_data" / "gfs.t00z.pgrb2.1p00.f024",
    _here.parent / "tests" / "input_data" / "gfs.t00z.pgrb2.1p00.f024",
]
grib2_file = next(str(p) for p in _candidates if p.exists())

gen = ReferenceGenerator(grib2_file)
manifest = gen.generate()
print(f"Manifest: {len(manifest['refs'])} refs for {grib2_file}")

# Persist for VirtualiZarr to read via URI
manifest_path = pathlib.Path(grib2_file).with_suffix(".json")
manifest_path.write_text(json.dumps(manifest))

TypeError: open_virtual_dataset() got an unexpected keyword argument 'engine'

## 2. Open as a VirtualiZarr Dataset

`open_virtual_dataset` reads the manifest and returns an `xarray.Dataset` backed by
`ManifestArray` objects — virtual metadata only, no data downloaded. This is useful
for inspecting structure, combining datasets, or generating consolidated manifests.


In [ ]:
manifest_url = manifest_path.resolve().as_uri()
registry = ObjectStoreRegistry(
    {
        manifest_url: LocalStore(prefix=str(manifest_path.parent)),
    }
)

vds = open_virtual_dataset(
    url=manifest_url,
    registry=registry,
    parser=KerchunkJSONParser(),
    loadable_variables=[],
)
display(vds)

## 3. Load Actual Data

A `ManifestArray` is virtual — calling `.compute()` on a VirtualiZarr dataset raises an
error. To load real data, pass the manifest to `open_dataset` (or use `open_grib2` to
skip the manifest step entirely).


In [ ]:
# open_dataset: manifest dict → xarray Dataset via Icechunk (local or remote)
ds = open_dataset(manifest)

# open_grib2: file path/URL → xarray Dataset in one call (same result)
# ds = open_grib2(grib2_file)

if "TMP" in ds.data_vars:
    data = ds.TMP.isel(y=slice(0, 10), x=slice(0, 10)).compute()
    print(f"TMP subset: shape={data.shape}, min={float(data.min()):.2f}, max={float(data.max()):.2f}")
    display(data)